### Decomposition of LWA Carrying Capacity into different contributions

This notebook follows the analysis by Barpanda et al. 2024 https://doi.org/10.1175/JCLI-D-24-0232.1 

We want to compute the Total LWA Carrying Capacity:
\begin{equation}
F_c(x) = \frac{C^2(x)}{4\alpha} = \frac{\left( u_0 + c_g^x - 2\alpha \mathcal{A}_0 \right)^2}{4\alpha}
\end{equation}
In order to do so we need to compute:

- $u_0 + c_g^x$: this is computed as a regression coefficient between 295 vertically-averaged linear zonal LWA flux $(𝐹_1 + 𝐹_3)$ and the wave-activity $(A cos\phi)$
- $\alpha$: this is the correlation coefficient between reference zonal winds and LWA. It can therefore be computed as −($u^′_e$$A^′_𝑇$ )/($A^′$$A^′_𝑇$ )
- ${A}_0$: this is the stationary waves induced local wave activity and can be computed by computing the local wave activity of the time-averaged qgpv field.

The Total LWA Carrying Capacity can therefore be decomposed in the following terms:

- 


In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.util as cutil
import numpy as np
from tqdm import tqdm
import matplotlib.gridspec as gridspec  # To control widths
from scipy.integrate import trapezoid

In [2]:
#here we focus on the asymmetric baseline simulation
%cd /home/links/mf671/DATA/ERA5/falwa_data/

/home/links/mf671/DATA/ERA5/falwa_data


In [3]:
#import dataset with fluxes (output of falwa)
data = xr.open_mfdataset('/home/links/mf671/DATA/ERA5/falwa_data/falwa_output_per_month/falwa_output_daily_1.5x1.5_*.nc',parallel=True)
ucomp_ERA5 = data['u_baro'] - data['dw_uref']
lwa_ERA5 = data['dw_lwa']
qgpv_ERA5 = data['dw_qgpv']
qref_ERA5 = data['dw_qref']
F1_ERA5 = data['adv_flux_f1'].copy()
F2_ERA5 = data['adv_flux_f2'].copy()
F3_ERA5 = data['adv_flux_f3'].copy()
lon = data['longitude']
lat = data['latitude']

print(len(data.valid_time))
data.close()

1805


### Necessary functions

In [4]:
from scipy.signal import butter, filtfilt

import xarray as xr
import numpy as np

import xarray as xr

import xarray as xr

def rolling_periodic(
    da: xr.DataArray,
    dim: str,
    window: int,
    center: bool = True
) -> xr.DataArray:

    n = window // 2 if center else window - 1

    # Extend array by wrapping values from both ends along chosen dimension
    da_extended = xr.concat([da.isel({dim: slice(-n, None)}),
                             da,
                             da.isel({dim: slice(0, n)})],
                            dim=dim)

    # Apply rolling mean
    rolled = da_extended.rolling({dim: window}, center=center).mean()

    # Trim back to original length
    result = rolled.isel({dim: slice(n, -n)})
    result[dim] = da[dim]  # restore original coords

    return result

def lowpass_fourier(da, period_days=4 , dt=1, time_dim='valid_time'):
    
    n_time = da.sizes[time_dim]

    # FFT frequencies (cycles per day)
    freqs = np.fft.fftfreq(n_time, d=dt)
    cutoff = 1 / period_days  # max frequency to keep

    # FFT along time axis
    fft_vals = np.fft.fft(da, axis=0)

    # Low-pass mask
    mask = np.abs(freqs) <= cutoff
    fft_vals_filtered = fft_vals * mask[:, np.newaxis, np.newaxis]

    # Inverse FFT
    filtered_data = np.fft.ifft(fft_vals_filtered, axis=0).real

    return xr.DataArray(filtered_data, dims=da.dims, coords=da.coords, name=da.name)

def longitude_running_avg(da, lon_window_deg=15, lon_dim='longitude'):
    # Compute number of points corresponding to lon_window_deg
    lon_vals = da[lon_dim].values
    if len(lon_vals) > 1:
        lon_res = np.mean(np.diff(lon_vals))
        window_points = max(1, int(np.round(lon_window_deg / lon_res)))
    else:
        window_points = 1

    print('longitudinal filter window: '+ str(window_points))

    # Apply rolling mean along longitude
    da_smoothed = rolling_periodic(da, lon_dim , window = window_points, center=True)
    return da_smoothed

def deseasonalize_daily(da):
    daily_clim = da.groupby("valid_time.dayofyear").mean("valid_time")
    anomalies = da.groupby("valid_time.dayofyear") - daily_clim    
    return anomalies

def deseasonalize_roll(da):
    running_mean = rolling_periodic(da=da,dim='valid_time',window=30,center=True)
    mean = da.mean(dim='valid_time')
    anomalies = da - running_mean + mean 
    return anomalies

def deseasonalize_harmonic(da, n_harmonics=3, time_dim='valid_time'):
    """
    Remove the seasonal cycle following Barpanda & Nakamura (2025, Sec. 3a):
    "we subtract the seasonal cycle from the daily averaged time series which
    is computed using the first three annual harmonics of the daily means of
    the entire dataset."

    Same input/output contract as `deseasonalize_roll`: takes a DataArray
    with a time dimension `time_dim`, returns anomalies with the overall
    time-mean added back on (so the output's long-term mean matches the
    input's, not zero).
    """
    # 1. Daily climatology: mean value for each day-of-year (1..365/366),
    #    computed over the full input period (matches "entire dataset").
    daily_clim = da.groupby(f"{time_dim}.dayofyear").mean(time_dim)

    # 2. Keep only the annual mean (freq 0) + the first n_harmonics
    #    (freq 1..n_harmonics cycles/year) of that climatology.
    axis = daily_clim.get_axis_num('dayofyear')
    n = daily_clim.sizes['dayofyear']

    fft_vals = np.fft.fft(daily_clim.values, axis=axis)

    mask = np.zeros(n, dtype=bool)
    mask[0] = True                      # annual mean
    for k in range(1, n_harmonics + 1):
        mask[k] = True                  # +k cycles/year
        mask[-k] = True                 # matching negative frequency

    mask_shape = [1] * fft_vals.ndim
    mask_shape[axis] = n
    fft_vals_filtered = fft_vals * mask.reshape(mask_shape)

    smooth_clim_vals = np.fft.ifft(fft_vals_filtered, axis=axis).real
    smooth_clim = xr.DataArray(
        smooth_clim_vals,
        dims=daily_clim.dims,
        coords=daily_clim.coords,
        name=da.name,
    )

    # 3. Map the smoothed seasonal cycle back onto the full time axis via
    #    day-of-year (vectorized indexing: output dims follow `doy`, i.e.
    #    time_dim, not 'dayofyear').
    doy = da[time_dim].dt.dayofyear
    seasonal_cycle = smooth_clim.sel(dayofyear=doy)

    # 4. Anomalies, with the overall time-mean added back — same convention
    #    as deseasonalize_roll, so it's a drop-in replacement.
    overall_mean = da.mean(dim=time_dim)
    anomalies = da - seasonal_cycle + overall_mean

    return anomalies

In [5]:
def compute_covariance_xr(A,B):
    
    #A = A.where(np.isfinite(A), 0)
    #B = B.where(np.isfinite(B), 0)
    #A = A.where(np.isnan(A), 0)
    #B = B.where(np.isnan(B), 0)
    
    # Compute means lazily (Dask-aware)
    A_mean = A.mean(dim='valid_time')
    B_mean = B.mean(dim='valid_time')
    
    # Compute anomalies lazily
    A_anom = A - A_mean
    B_anom = B - B_mean
    
    # Covariance calculation
    cov = (A_anom * B_anom).mean(dim='valid_time')
    return cov
        

## Computing $\alpha$

$\alpha$ measures the anticorrelation between u and A

In formula it is given by:

𝑢𝑒 ≈ −𝛼A, 𝛼 > 0.

Or:

𝛼 = −($u^′_e$$A^′_𝑇$ )/($A^′$$A^′_𝑇$ ), where A′, 𝑢′ 𝑒 are row matrices containing the time-filtered transient LWA and zonal wind, respectively.

In [6]:
print('applying filters')
lwa = longitude_running_avg(lowpass_fourier(deseasonalize_harmonic(lwa_ERA5)))
ucomp = longitude_running_avg(lowpass_fourier(deseasonalize_harmonic(ucomp_ERA5)))
cosphi = np.cos(np.deg2rad(lat)).broadcast_like(ucomp)

applying filters
longitudinal filter window: 10
longitudinal filter window: 10


In [7]:
print('computing covariances')
cov = compute_covariance_xr(lwa/cosphi,ucomp)
var = compute_covariance_xr(lwa/cosphi,lwa/cosphi)
alpha = (- cov/var).persist()

neg_cov = - cov.copy()
var_lwa = var.copy()
var_ucomp = compute_covariance_xr(ucomp,ucomp)

computing covariances


In [8]:
print('computing alternative alpha')
A2 = (lwa/cosphi)**2
cov = compute_covariance_xr(A2,F2)
var = compute_covariance_xr(A2,A2)

neg_cov = - cov.copy()
var_lwa = var.copy()
var_ucomp = compute_covariance_xr(F2,F2)

alpha_alt = (- cov/var).persist()

computing alternative alpha


NameError: name 'F2' is not defined

In [ ]:
print('computing zonal average')
da_avg = alpha.mean(axis=-1).values
alpha_zon_mean = np.broadcast_to(da_avg[:, np.newaxis], alpha.shape)

In [ ]:
import cartopy.feature as cfeature

fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(10, 4))

# Set extent: lon from -180 to 180, lat from 25 to 75
ax.set_extent([-180, 180, 0, 90], crs=ccrs.PlateCarree())

# Plot the filled contours
contour = ax.contourf(
    lon, lat, alpha,
    levels=np.arange(0,1,0.1),
    cmap='YlOrRd',
    extend='both',
    transform=ccrs.PlateCarree()
)

# Add coastlines and optional borders
ax.set_aspect(1.6)
ax.coastlines(color='black', linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)

# Add colorbar
cbar = plt.colorbar(contour, ax=ax, orientation='vertical', label='unitless',shrink=0.5)
plt.title(r'$\alpha$: wave-mean flow interaction strength')

plt.show()

In [ ]:
import cartopy.feature as cfeature

fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(10, 4))

# Set extent: lon from -180 to 180, lat from 25 to 75
ax.set_extent([-180, 180, 0, 90], crs=ccrs.PlateCarree())

# Plot the filled contours
contour = ax.contourf(
    lon, lat, alpha_alt,
    levels=np.arange(0,1,0.1)/10,
    cmap='YlOrRd',
    extend='both',
    transform=ccrs.PlateCarree()
)

# Add coastlines and optional borders
ax.set_aspect(1.6)
ax.coastlines(color='black', linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)

# Add colorbar
cbar = plt.colorbar(contour, ax=ax, orientation='vertical', label='unitless',shrink=0.5)
plt.title(r'$\alpha$: wave-mean flow interaction strength')

plt.show()


In [ ]:
import cartopy.feature as cfeature

fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(10, 4))

# Set extent: lon from -180 to 180, lat from 25 to 75
ax.set_extent([-180, 180, 0, 90], crs=ccrs.PlateCarree())

# Plot the filled contours
contour = ax.contourf(
    lon, lat, neg_cov/np.sqrt(var_lwa*var_ucomp),
    levels=np.arange(0,1,0.1),
    cmap='Blues',
    extend='both',
    transform=ccrs.PlateCarree()
)

# Add coastlines and optional borders
ax.set_aspect(1.6)
ax.coastlines(color='black', linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)

# Add colorbar
cbar = plt.colorbar(contour, ax=ax, orientation='vertical', label='unitless',shrink=0.5)
plt.title(r'$\alpha$: wave-mean flow interaction strength')

plt.show()


## Computing $u_{ref} + c_g$

$u_0 + c_g^x$ is computed as a regression coefficient between vertically-averaged linear zonal LWA flux $(𝐹_1 + 𝐹_3)$ and the wave-activity $(A cos\phi)$

In [ ]:
print('applying filters')
F1 = longitude_running_avg(lowpass_fourier(deseasonalize_harmonic(F1_ERA5)))
F2 = longitude_running_avg(lowpass_fourier(deseasonalize_harmonic(F2_ERA5)))
F3 = longitude_running_avg(lowpass_fourier(deseasonalize_harmonic(F3_ERA5)))
zonal_flux = (F1_ERA5 + F3_ERA5)
tot_zonal_flux = (F1_ERA5 + F2_ERA5 + F3_ERA5)

In [ ]:
print('computing covariances')
cov = compute_covariance_xr(lwa,zonal_flux)
var = compute_covariance_xr(lwa,lwa)
ds_zon_group_velocity = (cov/var).persist()

In [ ]:
print('performing zonal average')
da_avg = ds_zon_group_velocity.mean(axis=-1).values
GV_zon_mean = np.broadcast_to(da_avg[:, np.newaxis], ds_zon_group_velocity.shape)

In [ ]:
import cartopy.feature as cfeature

fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(10, 4))

# Set extent: lon from -180 to 180, lat from 25 to 75
ax.set_extent([-180, 180, 25, 75], crs=ccrs.PlateCarree())

# Plot the filled contours
contour = ax.contourf(
    lon,lat,ds_zon_group_velocity,
    levels=np.arange(0, 24, 2.5),
    cmap='jet',#'YlOrRd',
    extend='both',
    transform=ccrs.PlateCarree()
)

# Add coastlines and optional borders
ax.set_aspect(1.6)
ax.coastlines(color='black', linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)

# Add colorbar
cbar = plt.colorbar(contour, ax=ax, orientation='vertical', label='$m/s$',shrink=0.5)
plt.title('$(u_{ref} + c^0_g)$: Rossby waves group velocity')

plt.show()

## Computing $A_0$

The stationary wave component of A0 can be roughly estimated through its mean value. Otherwise a more refined method would be to exclude the grid points during which blocking is happening. In this way, the stationary waves are not influenced by blocking and it is easier to assess causality relationship. In this case we are sticking to the first way because we didn't compute blocking climatology for ERA5

In [ ]:
from math import pi
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt
from scipy.integrate import trapezoid
from tqdm import tqdm

# --- Parameters --- #
Earth_radius = 6.378e+6 # Earth's radius

# --- Load the zonal wind and QGPV at 240hPa --- #
QGPV_File = xr.open_dataset('/home/links/mf671/DATA/ERA5/falwa_data/qgpv_qref_monthly_djf_1980-2000.nc')

# --- Read in longitude and latitude arrays --- #
xlon = QGPV_File.longitude.values
ylat = QGPV_File.latitude.values
height = QGPV_File.height.values
time = QGPV_File.valid_time.values
clat = np.abs(np.cos(ylat*pi/180.)) # cosine latitude
nlon = xlon.size
nlat = ylat.size

# --- Parameters needed to use the module HN2015_LWA --- 
dphi = (ylat[2]-ylat[1])*pi/180. # Equal spacing between latitude grid points, in radian
area = 2.*pi*Earth_radius**2 *(np.cos(ylat[:,np.newaxis]*pi/180.)*dphi)/float(nlon) * np.ones((nlat,nlon))
area = np.abs(area) # To make sure area element is always positive (given floating point errors).

# --- Read in the absolute vorticity field from the netCDF file --- #
QGPV = QGPV_File.qgpv
Qref_input = QGPV_File.qref

p0 = 1000.                          # surface pressure [hPa]
kmax = 96                           # number of grid points for vertical extrapolatitudeion (dimension of height)
dz = 500.                          # differential height element
height = np.arange(0,kmax)*dz       # pseudoheight [m]
hh = 6745.                          # scale height
    
prefactor = np.array([np.exp(-z/hh) for z in height[1:]]).sum() # integrated sum of density from the level
                                                                #just above the ground (z=1km) to aloft
coeff = np.array([np.exp(-z/hh) for z in height[1:]])

print('averaging qgpv')
density_weighted_qgpv = np.sum(coeff[np.newaxis,:,np.newaxis,np.newaxis]*QGPV[:,1:,:,:],axis=1)/prefactor
print(density_weighted_qgpv.shape)

print('averaging qref')
density_weighted_qref = np.sum(coeff[np.newaxis,:,np.newaxis]*Qref_input[:,1:,:],axis=1)/prefactor
print(density_weighted_qref.shape)

# initialize LWA value
LWA = density_weighted_qgpv.copy()
LWA.values = np.zeros(density_weighted_qgpv.shape)

#METHOD 1

#my own classes
import sys
sys.path.append("/emmy-noether/home/mf671/github/hn2016_falwa/src/falwa/")
from wrapper import qgpv_eqlat_lwa

for i,t in tqdm(enumerate(time)):
    Qref_tmp, LWA_tmp = qgpv_eqlat_lwa(ylat=ylat, vort=density_weighted_qgpv.loc[t,:,:].values,area=area,dmu=Earth_radius*clat*dphi,n_points=30,qref=density_weighted_qref.loc[t,:].values)
    LWA.loc[t,:,:] = LWA_tmp
        
density_weighted_stat_lwa_1 = LWA.mean('valid_time')

In [ ]:
#METHOD 2

#my own classes
import sys
sys.path.append("/emmy-noether/home/mf671/github/hn2016_falwa/src/falwa/")
from wrapper import barotropic_input_qref_to_compute_lwa

def compute_dw_stat_lwa(dw_qgpv,dw_qref):
   # --- Parameters --- #
    Earth_radius = 6.378e+6 # Earth's radius
    # --- Read in longitude and latitude arrays --- #
    lon = dw_qgpv.longitude.values
    lat = dw_qgpv.latitude.values
    clat = np.abs(np.cos(lat*pi/180.)) # cosine latitude
    nlon = lon.size
    nlat = lat.size
    
    # --- Parameters needed to use the module HN2015_LWA --- 
    dphi = (lat[2]-lat[1])*pi/180. # Equal spacing between latitude grid points, in radian
    area = 2.*pi*Earth_radius**2 *(np.cos(lat[:,np.newaxis]*pi/180.)*dphi)/float(nlon) * np.ones((nlat,nlon))
    area = np.abs(area) # To make sure area element is always positive (given floating point errors).
    
    # --- Read in the absolute vorticity field from the netCDF file --- #
    density_weighted_qgpv = dw_qgpv.mean('valid_time')
    density_weighted_qref = dw_qref.mean('valid_time')
    
    return barotropic_input_qref_to_compute_lwa(ylat=lat, qref=density_weighted_qref, vort=density_weighted_qgpv, area=area, dmu=Earth_radius*clat*dphi)[0]

density_weighted_stat_lwa_2 = compute_dw_stat_lwa(density_weighted_qgpv,density_weighted_qref)

In [ ]:
import cartopy.feature as cfeature

fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(10, 4))

# Set extent: lon from -180 to 180, lat from 25 to 75
ax.set_extent([-180, 180, 25, 75], crs=ccrs.PlateCarree())

# Plot the filled contours
contour = ax.contourf(
    xlon,ylat,density_weighted_stat_lwa_2,
    levels=np.arange(0, 42, 4),
    cmap='YlOrRd',
    extend='both',
    transform=ccrs.PlateCarree()
)

# Add coastlines and optional borders
ax.set_aspect(1.6)
ax.coastlines(color='black', linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)

# Add colorbar
cbar = plt.colorbar(contour, ax=ax, orientation='vertical', label='$m/s$',shrink=0.5)
plt.title(r'$A_0 cos\phi$: quasi stationary lwa (DJF)')

plt.show()

##### Compute the CARRYING CAPACITY

In [ ]:
#TOTAL CARRYING CAPACITY
cosphi2 = cosphi[0,:,:].copy()
FC = cosphi2*((ds_zon_group_velocity - 2*alpha*density_weighted_stat_lwa_2)**2)/(4*alpha)

#(((ds_zon_group_velocity-2*alpha*density_weighted_stat_lwa_2)**2)*cosphi2) / (4*alpha)

fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(10, 4))

# Set extent: lon from -180 to 180, lat from 25 to 75
ax.set_extent([-180, 180, 20, 80], crs=ccrs.PlateCarree())

# Plot the filled contours
contour = ax.contourf(
    lon,lat,FC,
    levels=np.arange(0, 300, 30),
    cmap='YlOrRd',
    extend='both',
    transform=ccrs.PlateCarree()
)

# Add coastlines and optional borders
ax.set_aspect(1.6)
ax.coastlines(color='black', linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)

# Add colorbar
cbar = plt.colorbar(contour, ax=ax, orientation='vertical', label='$m^2/s^2$',shrink=0.5)
plt.title('Carrying capacity: ' + r'$F_c(x) = \frac{\left( u_0 + c_g^x - 2\alpha \mathcal{A}_0 \right)^2 cos \phi}{4\alpha}$')

plt.show()

In [ ]:
#Testing the fluxes approximation 

lwa_prime = (lwa-density_weighted_stat_lwa_2[np.newaxis,:,:])/cosphi

#COMPUTING F1
u0 = data['dw_uref'].mean('valid_time')
F1_approx = lwa_prime*u0*cosphi

#COMPUTING F2
F2_approx = -(lwa_prime**2)*cosphi*alpha_alt#alpha_zon_mean
#F2_approx = -lwa**2*alpha

#COMPUTING F3
cg = GV_zon_mean[:,0] - u0
F3_approx = lwa_prime*cg*cosphi

#COMPUTING STAT_LWA FLUX
FA0 = GV_zon_mean*density_weighted_stat_lwa_2*cosphi - alpha_zon_mean*(density_weighted_stat_lwa_2**2)*cosphi

In [ ]:
print(F2_approx.shape)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 4), subplot_kw={'projection': ccrs.PlateCarree()})

time_step = 100

# Plot F1_approx
axs[0].set_extent([-180, 180, 25, 75], crs=ccrs.PlateCarree())
contour0 = axs[0].contourf(
    lon, lat, F1_approx[time_step,:,:],
    levels=np.linspace(np.nanmin(F1_approx), np.nanmax(F1_approx), 15),
    cmap='YlOrRd',
    extend='both',
    transform=ccrs.PlateCarree()
)
axs[0].set_aspect(1.6)
axs[0].coastlines(color='black', linewidth=0.8)
axs[0].add_feature(cfeature.BORDERS, linewidth=0.5)
plt.colorbar(contour0, ax=axs[0], orientation='vertical', label='$m^2/s^2$', shrink=0.7)
axs[0].set_title('F1 Approximation')

# Plot F1_ERA5
axs[1].set_extent([-180, 180, 25, 75], crs=ccrs.PlateCarree())
contour1 = axs[1].contourf(
    lon, lat, F1[time_step,:,:],
    levels=np.linspace(np.nanmin(F1_approx), np.nanmax(F1_approx), 15),
    cmap='YlOrRd',
    extend='both',
    transform=ccrs.PlateCarree()
)
axs[1].set_aspect(1.6)
axs[1].coastlines(color='black', linewidth=0.8)
axs[1].add_feature(cfeature.BORDERS, linewidth=0.5)
plt.colorbar(contour1, ax=axs[1], orientation='vertical', label='$m^2/s^2$', shrink=0.7)
axs[1].set_title('F1')

plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 4), subplot_kw={'projection': ccrs.PlateCarree()})

time_step = 100

# Plot F2_approx
axs[0].set_extent([-180, 180, 25, 75], crs=ccrs.PlateCarree())
contour0 = axs[0].contourf(
    lon, lat, F2_approx[time_step,:,:],
    levels=np.linspace(np.nanmin(F2_approx), np.nanmax(F2_approx), 15),
    cmap='YlOrRd',
    extend='both',
    transform=ccrs.PlateCarree()
)
axs[0].set_aspect(1.6)
axs[0].coastlines(color='black', linewidth=0.8)
axs[0].add_feature(cfeature.BORDERS, linewidth=0.5)
plt.colorbar(contour0, ax=axs[0], orientation='vertical', label='$m^2/s^2$', shrink=0.7)
axs[0].set_title('F2 Approximation')

# Plot F1_ERA5
axs[1].set_extent([-180, 180, 25, 75], crs=ccrs.PlateCarree())
contour1 = axs[1].contourf(
    lon, lat, F2[time_step,:,:],
    levels=np.linspace(np.nanmin(F2_approx), np.nanmax(F2_approx), 15),
    cmap='YlOrRd',
    extend='both',
    transform=ccrs.PlateCarree()
)
axs[1].set_aspect(1.6)
axs[1].coastlines(color='black', linewidth=0.8)
axs[1].add_feature(cfeature.BORDERS, linewidth=0.5)
plt.colorbar(contour1, ax=axs[1], orientation='vertical', label='$m^2/s^2$', shrink=0.7)
axs[1].set_title('F2')

plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 4), subplot_kw={'projection': ccrs.PlateCarree()})

time_step = 200

# Plot F3_approx
axs[0].set_extent([-180, 180, 25, 75], crs=ccrs.PlateCarree())
contour0 = axs[0].contourf(
    lon, lat, F3_approx[time_step,:,:],
    levels=np.linspace(np.nanmin(F3_approx), np.nanmax(F3_approx), 15),
    cmap='YlOrRd',
    extend='both',
    transform=ccrs.PlateCarree()
)
axs[0].set_aspect(1.6)
axs[0].coastlines(color='black', linewidth=0.8)
axs[0].add_feature(cfeature.BORDERS, linewidth=0.5)
plt.colorbar(contour0, ax=axs[0], orientation='vertical', label='$m^2/s^2$', shrink=0.7)
axs[0].set_title('F3 Approximation')

# Plot F1_ERA5
axs[1].set_extent([-180, 180, 25, 75], crs=ccrs.PlateCarree())
contour1 = axs[1].contourf(
    lon, lat, F3[time_step,:,:],
    levels=np.linspace(np.nanmin(F3_approx), np.nanmax(F3_approx), 15),
    cmap='YlOrRd',
    extend='both',
    transform=ccrs.PlateCarree()
)
axs[1].set_aspect(1.6)
axs[1].coastlines(color='black', linewidth=0.8)
axs[1].add_feature(cfeature.BORDERS, linewidth=0.5)
plt.colorbar(contour1, ax=axs[1], orientation='vertical', label='$m^2/s^2$', shrink=0.7)
axs[1].set_title('F3')

plt.tight_layout()
plt.show()

In [ ]:
#TRANSIENT FLUX computation
#CLIM_tot_zonal_flux = (F1_ERA5 + F2_ERA5 + F3_ERA5).mean('valid_time')
#CLIM_tot_zonal_flux = tot_zonal_flux.mean('valid_time')
#CLIM_stat_wave_zonal_flux = ds_zon_group_velocity*density_weighted_stat_lwa_2 - alpha*(density_weighted_stat_lwa_2)**2
transient_flux = F1_approx + F2_approx + F3_approx# - FA0
transient_flux_real = F1 + F2 + F3# - FA0*10
CLIM_transient_flux = transient_flux.mean('valid_time')
PERC_transient_flux = transient_flux.quantile(0.60, dim='valid_time')

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(10, 4))

# Set extent: lon from -180 to 180, lat from 25 to 75
ax.set_extent([-180, 180, 25, 75], crs=ccrs.PlateCarree())

# Plot the filled contours
contour = ax.contourf(
    lon,lat,CLIM_transient_flux,
    levels=np.arange(-420, 430, 40),
    cmap='Blues',
    extend='both',
    transform=ccrs.PlateCarree()
)

# Add coastlines and optional borders
ax.set_aspect(1.6)
ax.coastlines(color='black', linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)

# Add colorbar
cbar = plt.colorbar(contour, ax=ax, orientation='vertical', label='$m^2/s^2$',shrink=0.5)
plt.title('Snapshot of the transient zonal flux')

plt.show()

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(10, 4))

# Set extent: lon from -180 to 180, lat from 25 to 75
ax.set_extent([-180, 180, 25, 75], crs=ccrs.PlateCarree())

# Plot the filled contours
contour = ax.contourf(
    lon,lat,transient_flux_real.mean('valid_time'),
    levels=np.arange(-420, 430, 40)*5,
    cmap='Blues',
    extend='both',
    transform=ccrs.PlateCarree()
)

# Add coastlines and optional borders
ax.set_aspect(1.6)
ax.coastlines(color='black', linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)

# Add colorbar
cbar = plt.colorbar(contour, ax=ax, orientation='vertical', label='$m^2/s^2$',shrink=0.5)
plt.title('Snapshot of the transient zonal flux')

plt.show()

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(10, 4))

# Set extent: lon from -180 to 180, lat from 25 to 75
ax.set_extent([-180, 180, 20, 80], crs=ccrs.PlateCarree())

# Plot the filled contours
contour = ax.contourf(
    lon,lat,PERC_transient_flux-FC,
    levels=np.arange(-420, 420, 40),
    cmap='bwr',
    extend='both',
    transform=ccrs.PlateCarree()
)

# Add coastlines and optional borders
ax.set_aspect(1.6)
ax.coastlines(color='black', linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)

# Add colorbar
cbar = plt.colorbar(contour, ax=ax, orientation='vertical', label='$m^2/s^2$',shrink=0.5)
plt.title('Difference')

plt.show()